# AIC 2026 - Final T4x2 self-cut smoke

Attach `lyduchoang/aic-26-video` and `khoalequangminh/aic-test-dataset`, select **GPU T4 x2**, enable Internet, and enable Kaggle Secrets `AIC_RCLONE_CONFIG` plus `AIC_GDRIVE_FOLDER_ID`.

This notebook runs TransNetV2 for `L21_V001` and `L21_V002` in parallel, extracts every adaptive frame by exact zero-based frame ordinal, builds a BTC-compatible package, validates it, previews it, and uploads it with rclone. It does not run DAM, OCR, ASR, embeddings, or repository tests.


In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
BRANCH = "feat/offline-frame-extraction-transnetv2"
TARGET = Path("/kaggle/working/AIC-2026")

def run_streamed(command, *, cwd=None, env=None, prefix=""):
    print("$", " ".join(map(str, command)), flush=True)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    assert process.stdout is not None
    lines = []
    for line in process.stdout:
        print(f"{prefix}{line}", end="", flush=True)
        lines.append(line.rstrip())
    return_code = process.wait()
    if return_code != 0:
        tail = "\n".join(lines[-80:])
        raise RuntimeError(f"Command failed with code {return_code}: {' '.join(map(str, command))}\n{tail}")
    return lines

clone_env = os.environ.copy()
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret("AIC_GITHUB_TOKEN")
except Exception:
    github_token = None
if github_token:
    clone_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Bearer {github_token}",
    })

print(f"[repo] phase=sync result=attempting branch={BRANCH}", flush=True)
if TARGET.exists() and not (TARGET / ".git").is_dir():
    raise RuntimeError(f"Target exists but is not a git repository: {TARGET}")
if not TARGET.exists():
    run_streamed(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(TARGET)], env=clone_env)
else:
    run_streamed(["git", "fetch", "origin", BRANCH], cwd=TARGET, env=clone_env)
    run_streamed(["git", "switch", BRANCH], cwd=TARGET, env=clone_env)
    run_streamed(["git", "pull", "--ff-only", "origin", BRANCH], cwd=TARGET, env=clone_env)
os.chdir(TARGET)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"[repo] phase=sync result=success branch={branch} commit={commit}", flush=True)


In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys

VIDEO_IDS = ["L21_V001", "L21_V002"]
VIDEO_DIR = Path("/kaggle/input/datasets/lyduchoang/aic-26-video/Video/Video/Videos_L21_a/video")
VIDEO_PATHS = {video_id: VIDEO_DIR / f"{video_id}.mp4" for video_id in VIDEO_IDS}
MAP_V001 = Path("/kaggle/input/datasets/khoalequangminh/aic-test-dataset/data/map-keyframes/L21_V001.csv")
ARTIFACT_ROOT = Path("/kaggle/working/aic2026-artifacts")
PACKAGE_NAME = "self-cut-btc-compatible"
PACKAGE_ROOT = ARTIFACT_ROOT / "exports" / PACKAGE_NAME
BATCH_SIZE = 16

os.environ.update({
    "AIC_DATA_ROOT": "/kaggle/input",
    "AIC_ARTIFACT_ROOT": str(ARTIFACT_ROOT),
    "AIC_PACKAGE_NAME": PACKAGE_NAME,
    "AIC_TRANSNETV2_BACKEND": "pytorch",
    "AIC_EXPORT_GDRIVE": "1",
    "AIC_GDRIVE_BACKEND": "rclone",
})

print("[preflight] phase=inputs result=attempting", flush=True)
for video_id, path in VIDEO_PATHS.items():
    print(f"[preflight] video_id={video_id} path={path} exists={path.is_file()}", flush=True)
    if not path.is_file():
        raise FileNotFoundError(f"Required benchmark video is missing: {path}")
if not MAP_V001.is_file():
    raise FileNotFoundError(f"Organizer calibration map is missing: {MAP_V001}")
for binary in ("git", "ffmpeg", "ffprobe", "nvidia-smi"):
    resolved = shutil.which(binary)
    print(f"[preflight] binary={binary} path={resolved or '<missing>'}", flush=True)
    if not resolved:
        raise RuntimeError(f"Required binary is missing: {binary}")

import torch
print(f"[preflight] torch={torch.__version__} cuda={torch.cuda.is_available()} gpu_count={torch.cuda.device_count()}", flush=True)
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError("This final smoke requires Kaggle Accelerator = GPU T4 x2")
for index in range(2):
    name = torch.cuda.get_device_name(index)
    print(f"[preflight] gpu={index} name={name}", flush=True)
    if "T4" not in name.upper():
        raise RuntimeError(f"GPU {index} is not a Tesla T4: {name}")

stale = [ARTIFACT_ROOT / "shot_detection" / f"{video_id}.jsonl" for video_id in VIDEO_IDS]
stale = [path for path in stale if path.exists()]
if stale:
    raise RuntimeError(f"Clean final run required. Delete old artifacts first: {stale}")

from aic2026.common.frame_manifest import read_frame_map
with MAP_V001.open("r", encoding="utf-8-sig", newline="") as stream:
    raw_rows = list(csv.DictReader(stream))
parsed_rows = read_frame_map(MAP_V001)
raw_indices = [int(row["frame_idx"]) for row in raw_rows]
parsed_indices = [row.frame_idx for row in parsed_rows]
if raw_indices != parsed_indices:
    raise AssertionError("Pipeline changed organizer frame_idx values")
round_differences = sum(
    round(float(row["pts_time"]) * float(row["fps"])) != int(row["frame_idx"])
    for row in raw_rows
)
floor_matches = sum(
    int(float(row["pts_time"]) * float(row["fps"])) == int(row["frame_idx"])
    for row in raw_rows
)
if floor_matches != len(raw_rows) or round_differences != 74:
    raise AssertionError(
        f"Unexpected organizer calibration: floor_matches={floor_matches}, "
        f"round_differences={round_differences}"
    )
print(
    f"[preflight] phase=organizer_index result=success rows={len(raw_rows)} "
    f"preserved={len(parsed_indices)} floor_matches={floor_matches} round_differences={round_differences}",
    flush=True,
)
print("[preflight] status=ready", flush=True)


## Google Drive preflight via rclone

`AIC_RCLONE_CONFIG` must contain `base64:<rclone.conf payload>`; `AIC_GDRIVE_FOLDER_ID` is the parent folder ID. Credentials are never printed.


In [ ]:
import base64
import binascii
import configparser
import os
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
rclone_secret = (secrets.get_secret("AIC_RCLONE_CONFIG") or "").strip()
gdrive_folder_id = (secrets.get_secret("AIC_GDRIVE_FOLDER_ID") or "").strip()
if not rclone_secret or not gdrive_folder_id:
    raise ValueError("Enable non-empty AIC_RCLONE_CONFIG and AIC_GDRIVE_FOLDER_ID secrets")
if rclone_secret.startswith("base64:"):
    try:
        rclone_text = base64.b64decode(
            "".join(rclone_secret.removeprefix("base64:").split()), validate=True
        ).decode("utf-8")
    except (binascii.Error, UnicodeDecodeError, ValueError) as error:
        raise ValueError("AIC_RCLONE_CONFIG contains invalid base64") from error
else:
    rclone_text = rclone_secret
parsed = configparser.RawConfigParser()
parsed.read_string(rclone_text)
RCLONE_REMOTE = os.environ.get("AIC_RCLONE_REMOTE", "gdrive")
if not parsed.has_section(RCLONE_REMOTE) or parsed.get(RCLONE_REMOTE, "type", fallback="") != "drive":
    raise ValueError(f"rclone config must contain a [{RCLONE_REMOTE}] drive remote")

RCLONE_CONFIG = Path("/tmp/aic2026-rclone/rclone.conf")
RCLONE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
RCLONE_CONFIG.write_text(rclone_text, encoding="utf-8")
RCLONE_CONFIG.chmod(0o600)
RCLONE_BIN = Path("/kaggle/working/bin/rclone")
if not RCLONE_BIN.is_file():
    archive = Path("/tmp/rclone.zip")
    print("[rclone] phase=install result=attempting", flush=True)
    urllib.request.urlretrieve("https://downloads.rclone.org/rclone-current-linux-amd64.zip", archive)
    with zipfile.ZipFile(archive) as bundle:
        member = next(item for item in bundle.infolist() if item.filename.endswith("/rclone"))
        RCLONE_BIN.parent.mkdir(parents=True, exist_ok=True)
        RCLONE_BIN.write_bytes(bundle.read(member))
    RCLONE_BIN.chmod(0o755)
print(f"[rclone] phase=install result=success path={RCLONE_BIN}", flush=True)

preflight = [
    sys.executable, "-u", "scripts/export_frame_artifacts_with_rclone.py",
    "--rclone-bin", str(RCLONE_BIN), "--config", str(RCLONE_CONFIG),
    "--remote", RCLONE_REMOTE, "--root-folder-id", gdrive_folder_id,
    "--remote-root-name", PACKAGE_NAME, "--preflight-only",
]
run_streamed(preflight)
print("[rclone] phase=preflight result=success", flush=True)


## TransNetV2 PyTorch runtime and pinned checkpoint


In [ ]:
import hashlib
import os
import subprocess
import urllib.request
from pathlib import Path

TRANSNET_SOURCE = Path("/kaggle/working/TransNetV2-source")
PYTORCH_MODULE = TRANSNET_SOURCE / "inference-pytorch" / "transnetv2_pytorch.py"
WEIGHTS = Path("/kaggle/working/transnetv2-pytorch/transnetv2-pytorch-weights.pth")
WEIGHTS_URL = "https://huggingface.co/ByteDance/shot2story/resolve/ff853c571fd92eb4e0c5713e27f2a323ac903f67/transnetv2-pytorch-weights.pth?download=true"
WEIGHTS_SHA256 = "a313d0b3bebfa9a71914b375bfdf918a30b5c3b1e6be51972d35dd8078b442de"

if not PYTORCH_MODULE.is_file():
    clone_env = os.environ.copy()
    clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
    run_streamed(["git", "clone", "--depth=1", "https://github.com/soCzech/TransNetV2.git", str(TRANSNET_SOURCE)], env=clone_env)
if not PYTORCH_MODULE.is_file():
    raise FileNotFoundError(f"TransNetV2 PyTorch module is missing: {PYTORCH_MODULE}")

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not WEIGHTS.is_file() or sha256(WEIGHTS) != WEIGHTS_SHA256:
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    partial = WEIGHTS.with_suffix(".partial")
    print("[transnet_setup] checkpoint=downloading", flush=True)
    urllib.request.urlretrieve(WEIGHTS_URL, partial)
    if sha256(partial) != WEIGHTS_SHA256:
        raise ValueError("Downloaded TransNetV2 checkpoint SHA-256 mismatch")
    os.replace(partial, WEIGHTS)
print(f"[transnet_setup] module={PYTORCH_MODULE}", flush=True)
print(f"[transnet_setup] checkpoint={WEIGHTS} sha256=verified", flush=True)


## Dual-GPU TransNetV2 benchmark

Both workers start together. Logs are prefixed by video ID, while `nvidia-smi` reports utilization and memory every 10 seconds.


In [ ]:
import json
import os
import subprocess
import sys
import threading
import time
from pathlib import Path

commands = {}
for gpu_index, video_id in enumerate(VIDEO_IDS):
    commands[video_id] = [
        sys.executable, "-u", "scripts/run_transnetv2_shots.py",
        "--config", "configs/offline/frame_extraction.yaml",
        "--backend", "pytorch", "--batch-size", str(BATCH_SIZE),
        "--video-id", video_id, "--video-path", str(VIDEO_PATHS[video_id]),
        "--output-root", str(ARTIFACT_ROOT), "--entrypoint", str(PYTORCH_MODULE),
        "--weights", str(WEIGHTS), "--resume",
    ]

processes = {}
captured = {video_id: [] for video_id in VIDEO_IDS}
started_at = time.monotonic()
for gpu_index, video_id in enumerate(VIDEO_IDS):
    environment = os.environ.copy()
    environment["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
    environment["PYTHONUNBUFFERED"] = "1"
    print(f"[benchmark] launching video_id={video_id} physical_gpu={gpu_index}", flush=True)
    print("$", " ".join(commands[video_id]), flush=True)
    processes[video_id] = subprocess.Popen(
        commands[video_id], env=environment, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )

completed_at = {}
def forward(video_id):
    process = processes[video_id]
    assert process.stdout is not None
    for line in process.stdout:
        captured[video_id].append(line.rstrip())
        print(f"[{video_id}] {line}", end="", flush=True)
    completed_at[video_id] = time.monotonic()

threads = [threading.Thread(target=forward, args=(video_id,), daemon=True) for video_id in VIDEO_IDS]
for thread in threads:
    thread.start()

while any(process.poll() is None for process in processes.values()):
    report = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=False,
    )
    for line in report.stdout.splitlines():
        print(f"[gpu_monitor] {line}", flush=True)
    time.sleep(10)
for thread in threads:
    thread.join()
failed = {video_id: process.returncode for video_id, process in processes.items() if process.returncode != 0}
if failed:
    tails = {video_id: lines[-80:] for video_id, lines in captured.items()}
    raise RuntimeError(f"Dual-GPU benchmark failed: {failed}\n{json.dumps(tails, indent=2)}")

wall_elapsed_s = max(completed_at.values()) - started_at
metrics = {}
aggregate_video_s = 0.0
for video_id in VIDEO_IDS:
    metrics_path = ARTIFACT_ROOT / "shot_detection" / "transnetv2_work" / video_id / "transnetv2_metrics.json"
    value = json.loads(metrics_path.read_text(encoding="utf-8"))
    shot_path = ARTIFACT_ROOT / "shot_detection" / f"{video_id}.jsonl"
    first_shot = json.loads(shot_path.read_text(encoding="utf-8").splitlines()[0])
    video_s = value["decoded_frames"] / first_shot["fps"]
    value["video_duration_s"] = video_s
    value["realtime_factor"] = video_s / value["total_elapsed_s"]
    aggregate_video_s += video_s
    metrics[video_id] = value

BENCHMARK_SUMMARY = {
    "status": "completed", "videos": metrics, "wall_elapsed_s": wall_elapsed_s,
    "aggregate_video_s": aggregate_video_s,
    "aggregate_realtime_factor": aggregate_video_s / wall_elapsed_s,
    "batch_size": BATCH_SIZE,
}
benchmark_dir = PACKAGE_ROOT / "benchmark"
benchmark_dir.mkdir(parents=True, exist_ok=True)
(benchmark_dir / "t4x2_summary.json").write_text(json.dumps(BENCHMARK_SUMMARY, indent=2) + "\n", encoding="utf-8")
print(json.dumps(BENCHMARK_SUMMARY, indent=2), flush=True)


## Build all adaptive frames and BTC-compatible package

No organizer images and no 0.5-second deduplication are used. Each video is decoded once for exact-index extraction.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

for video_id in VIDEO_IDS:
    shots = ARTIFACT_ROOT / "shot_detection" / f"{video_id}.jsonl"
    candidates = ARTIFACT_ROOT / "frame_extraction" / "adaptive_candidates" / f"{video_id}.jsonl"
    run_streamed([
        sys.executable, "-u", "scripts/build_adaptive_frame_candidates.py",
        "--video-id", video_id, "--shots", str(shots),
        "--output-root", str(ARTIFACT_ROOT),
    ], prefix=f"[{video_id}] ")
    run_streamed([
        sys.executable, "-u", "scripts/extract_adaptive_frames.py",
        "--video-id", video_id, "--video-path", str(VIDEO_PATHS[video_id]),
        "--candidates", str(candidates), "--output-root", str(ARTIFACT_ROOT),
    ], prefix=f"[{video_id}] ")
    run_streamed([
        sys.executable, "-u", "scripts/build_transnet_keyframe_package.py",
        "--video-id", video_id, "--output-root", str(ARTIFACT_ROOT),
        "--package-root", str(PACKAGE_ROOT),
    ], prefix=f"[{video_id}] ")

validation_lines = run_streamed([
    sys.executable, "-u", "scripts/validate_keyframe_package.py",
    "--package-root", str(PACKAGE_ROOT),
    "--video-id", VIDEO_IDS[0], "--video-id", VIDEO_IDS[1],
])
PACKAGE_REPORT = json.loads(next(line for line in reversed(validation_lines) if line.startswith("{")))
print(json.dumps(PACKAGE_REPORT, indent=2), flush=True)


## Timeline preview


In [ ]:
import csv
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import Markdown, display

def timecode(seconds):
    total = round(float(seconds) * 1000)
    return f"{total // 3600000:02d}:{(total // 60000) % 60:02d}:{(total // 1000) % 60:02d}.{total % 1000:03d}"

for video_id in VIDEO_IDS:
    batch = video_id.split("_", 1)[0]
    frames_dir = PACKAGE_ROOT / f"Keyframes_{batch}" / "keyframes" / video_id
    map_path = PACKAGE_ROOT / "map-keyframes" / f"{video_id}.csv"
    with map_path.open("r", encoding="utf-8", newline="") as stream:
        rows = list(csv.DictReader(stream))
    count = min(12, len(rows))
    positions = sorted({round(index * (len(rows) - 1) / max(1, count - 1)) for index in range(count)})
    panels = []
    for position in positions:
        row = rows[position]
        image = Image.open(frames_dir / f"{int(row['n']):06d}.jpg").convert("RGB")
        image.thumbnail((320, 180))
        panel = Image.new("RGB", (320, 215), "white")
        panel.paste(image, ((320 - image.width) // 2, 0))
        draw = ImageDraw.Draw(panel)
        draw.text((8, 184), f"n={row['n']}  {timecode(row['pts_time'])}", fill="black")
        draw.text((8, 199), f"frame_idx={row['frame_idx']}", fill="black")
        panels.append(panel)
    sheet = Image.new("RGB", (640, ((len(panels) + 1) // 2) * 215), "white")
    for index, panel in enumerate(panels):
        sheet.paste(panel, ((index % 2) * 320, (index // 2) * 215))
    display(Markdown(f"### {video_id}: {len(rows)} self-cut frames"))
    display(sheet)


## Upload package to Google Drive


In [ ]:
import json
import sys

upload_lines = run_streamed([
    sys.executable, "-u", "scripts/export_frame_artifacts_with_rclone.py",
    "--rclone-bin", str(RCLONE_BIN), "--config", str(RCLONE_CONFIG),
    "--remote", RCLONE_REMOTE, "--root-folder-id", gdrive_folder_id,
    "--remote-root-name", PACKAGE_NAME, "--package-root", str(PACKAGE_ROOT),
])
UPLOAD_REPORT = json.loads(next(line for line in reversed(upload_lines) if line.startswith("{")))
print("[final] upload_report=", json.dumps(UPLOAD_REPORT, indent=2), flush=True)
print("[final] package_root=", PACKAGE_ROOT, flush=True)
print("[final] benchmark=", PACKAGE_ROOT / "benchmark" / "t4x2_summary.json", flush=True)
print("[final] status=completed", flush=True)
